In [ ]:
import pickle
import shutil
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject, transform_bounds
from rasterio.windows import from_bounds, Window
from rasterio.crs import CRS

# ------------------------------
# 1. Paths
# ------------------------------
input_dir = Path(r"C:\Users\steph\Downloads\master_thesis\exports\object_classification_results")
output_dir = input_dir / "B_files_combined"
output_dir.mkdir(exist_ok=True)

output_tif = output_dir / "combined_classification_10m.tif"

ref_tif = Path(r"C:\Users\steph\Downloads\master_thesis\exports\sentinel2_fire_images\classification_report\random_forest\fire_04_random_forest_classification.tif")

# ------------------------------
# 2. Target CRS from reference
# ------------------------------
with rasterio.open(ref_tif) as ref:
    target_crs = ref.crs
    if target_crs is None:
        raise ValueError("Reference file has no CRS! Set manually, e.g. CRS.from_epsg(32633)")
print(f"Target CRS: {target_crs}")

# ------------------------------
# 3. Collect input TIFs and class names
# ------------------------------
tif_paths = []
class_names = None

for subdir in input_dir.glob("B*"):
    if not subdir.is_dir() or subdir == output_dir:
        continue
    tif_files = list(subdir.glob("*_obia_true.tif"))
    if not tif_files:
        print(f"Warning: No TIF in {subdir}")
        continue
    tif_paths.append(tif_files[0])
    if class_names is None:
        pkl_files = list(subdir.glob("*_class_names.pkl"))
        if pkl_files:
            with open(pkl_files[0], "rb") as f:
                class_names = pickle.load(f)
            print(f"Loaded classes from {pkl_files[0].name}")

if not tif_paths:
    raise RuntimeError("No TIF files found.")
print(f"\nFound {len(tif_paths)} tiles.")

# ------------------------------
# 4. Union bounds in target CRS
# ------------------------------
all_bounds = []
for path in tif_paths:
    with rasterio.open(path) as src:
        src_crs = src.crs if src.crs is not None else target_crs
        bounds = transform_bounds(src_crs, target_crs, *src.bounds)
        all_bounds.append(bounds)

union_left   = min(b[0] for b in all_bounds)
union_bottom = min(b[1] for b in all_bounds)
union_right  = max(b[2] for b in all_bounds)
union_top    = max(b[3] for b in all_bounds)

res = 10.0
output_width  = int(np.ceil((union_right - union_left) / res)) + 10   # add buffer
output_height = int(np.ceil((union_top - union_bottom) / res)) + 10
out_transform = rasterio.transform.from_origin(union_left, union_top, res, res)

print(f"Output size: {output_width} x {output_height}")
print(f"Output bounds: left={union_left}, bottom={union_bottom}, right={union_right}, top={union_top}")

# ------------------------------
# 5. Create empty output GeoTIFF
# ------------------------------
out_meta = {
    "driver": "GTiff",
    "height": output_height,
    "width": output_width,
    "count": 1,
    "dtype": "uint8",
    "crs": target_crs,
    "transform": out_transform,
    "compress": "lzw",
    "nodata": 0,
    "BIGTIFF": "YES",
    "tiled": True,
    "blockxsize": 512,
    "blockysize": 512,
}
with rasterio.open(output_tif, "w", **out_meta) as dst:
    pass
print("Empty output file created.\n")

# ------------------------------
# 6. Reproject each tile into the mosaic
# ------------------------------
for i, path in enumerate(tif_paths, start=1):
    print(f"Processing tile {i}/{len(tif_paths)}: {path.name}")
    with rasterio.open(path) as src:
        src_crs = src.crs if src.crs is not None else target_crs
        if src.crs is None:
            print("  (CRS was None, using target CRS)")

        data = src.read(1)
        src_transform = src.transform

        if src_transform.e > 0:   # north‑down → flip
            print("  Flipping north‑down tile")
            data = np.flipud(data)
            src_transform = rasterio.Affine(
                src_transform.a, src_transform.b, src_transform.c,
                src_transform.d, -src_transform.e,
                src_transform.f + src.height * src_transform.e
            )

        src_bounds = rasterio.transform.array_bounds(src.height, src.width, src_transform)
        target_bounds = transform_bounds(src_crs, target_crs, *src_bounds)

        # Window in output pixel coordinates
        window = from_bounds(*target_bounds, transform=out_transform)
        window = window.round_offsets(op='floor', pixel_precision=0)
        window = window.round_lengths(op='ceil', pixel_precision=0)

        with rasterio.open(output_tif, "r+") as dst:
            out_window = Window(0, 0, dst.width, dst.height)
            try:
                window = window.intersection(out_window)
            except rasterio.errors.WindowError:
                print(f"  Warning: {path.name} outside output bounds – skipped.")
                continue

            if window.width <= 0 or window.height <= 0:
                print(f"  Tile outside output bounds – skipped.")
                continue

            dst_array = np.zeros((window.height, window.width), dtype=data.dtype)
            dst_transform = rasterio.windows.transform(window, out_transform)

            reproject(
                source=data,
                destination=dst_array,
                src_transform=src_transform,
                src_crs=src_crs,
                dst_transform=dst_transform,
                dst_crs=target_crs,
                resampling=Resampling.nearest,
            )

            dst.write(dst_array, 1, window=window)
            print(f"  Written window {window}")

print(f"\nMerged raster saved to: {output_tif}")

# ------------------------------
# 7. Validation sample
# ------------------------------
with rasterio.open(output_tif) as dst:
    sample = dst.read(1, window=Window(0,0,100,100))
    print(f"Sample min: {sample.min()}, max: {sample.max()}")

# Clean up temp
temp_dir = tempfile.mkdtemp()
shutil.rmtree(temp_dir, ignore_errors=True)

# ------------------------------
# 8. Visualise (down‑sampled)
# ------------------------------
with rasterio.open(output_tif) as dst:
    downsample = 100
    ds_height = dst.height // downsample
    ds_width  = dst.width // downsample
    data = dst.read(1, out_shape=(ds_height, ds_width), resampling=Resampling.nearest)

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(data, cmap="tab20", interpolation="none")

if class_names:
    unique_classes = np.unique(data)
    unique_classes = unique_classes[unique_classes != 0]
    colors = [im.cmap(im.norm(cls)) for cls in unique_classes]
    patches = [mpatches.Patch(color=colors[i],
                              label=f"{cls}: {class_names.get(cls, 'unknown')}")
               for i, cls in enumerate(unique_classes)]
    ax.legend(handles=patches, bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()

ax.set_title(f"Merged Classification (10 m) – down‑sampled {downsample}×")
ax.set_xlabel("Column (pixel)")
ax.set_ylabel("Row (pixel)")
plt.show()